# Review controls and retrieval baselines

Run after completed notebooks 00–16. This does not replace their outputs.
Default: Continuous (`clip`) only, the original ECG cohort (up to 250), and an outcome-blind 100-examination rendering subset.
The raw ECG arm is cosine retrieval of frozen foundation features, with **no trimodal training**.
The bridge ECG-only arm has trimodal-trained features but ECG-only scoring. Do not conflate these controls.
Broken linkage permutes CXR/lab bundles across patients, retaining ECG anchors and the global bundle multiset.
It deliberately conceals the permutation from the generator; only this research control has synthetic linkage.
All controls share the same eligible training admissions and k=2. No test-label tuning.
Freeze settings before generation. Source manifests and feature hashes are recorded.
Changing the cohort/methods/code later requires a fresh E4_REVIEW_ROOT.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get("EXPERIMENT4_CODE_ROOT", str(Path.cwd())))
if not (CODE_ROOT / "e4_review_controls.py").is_file():
    CODE_ROOT = Path.cwd()
if not (CODE_ROOT / "e4_review_controls.py").is_file():
    raise FileNotFoundError("Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 code folder")
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
os.environ.setdefault("EXPERIMENT4_ROOT", str(Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", str(CODE_ROOT.parent / "artifacts"))) / "experiment4"))
os.environ.setdefault("HF_HOME", str(CODE_ROOT.parent / ".cache" / "huggingface"))
from e4_review_controls import roots
SOURCE_ROOT, REVIEW_ROOT = roots()
print({"source_read_only": str(SOURCE_ROOT), "new_output": str(REVIEW_ROOT)})


In [ ]:
from e4_review_controls import prepare_controls
protocol = prepare_controls(
    methods=tuple(os.environ.get("E4_REVIEW_METHODS", "clip").split(",")),
    max_queries=int(os.environ.get("E4_REVIEW_QUERY_N", "250")),
    render_subset=int(os.environ.get("E4_REVIEW_RENDER_N", "100")))
protocol


## Immediate no-generator baseline
Copy the nearest neighbor's machine interpretation exactly. This is a reference-transfer floor,
not a clinically acceptable report. It must be compared on common examinations with generated reports.
Partial status is expected here until notebooks 19 and 20 finish.

In [ ]:
from e4_review_controls import evaluate
evaluate(allow_partial=True)
